# Kronos-base — IDX 15-Minute Production Fine-Tuning (80 GB GPU)

Pipeline intraday terpisah dengan profil optimizer yang sama seperti varian
harian 80 GB. Context 240 bar mewakili kira-kira 12 sesi IDX dan forecast
20 bar mewakili kira-kira satu sesi. Tokenizer pretrained tetap dibekukan.
July validation diikuti clean production refit, sama seperti template harian.

## 1. Kaggle setup

Aktifkan **GPU** dan **Internet** pada Kaggle. Jika source Kronos tidak ikut
ter-clone bersama repository utama, sel setup akan mengambil source resmi dan
mengunci commit yang telah diuji.

In [ ]:
# Blackwell (sm_120) requires a PyTorch binary built with CUDA 12.8+.
# After this install completes, restart the Kaggle kernel once, then Run All.
%pip install -q --upgrade torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
%pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 pyarrow plotly kaleido tqdm

In [ ]:
from pathlib import Path
import os, sys, math, json, random, shutil, subprocess, warnings
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # Blackwell memakai CUDA 12.8 build; izinkan PyTorch memilih SDPA tercepat.
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)

def find_data():
    roots = [Path.cwd(), Path("/content"), Path("/kaggle/working"), Path("/kaggle/input")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / "Kronos IDX FineTune 15 Minutes" / "data" / "idx_kronos_all_15m.parquet"
        if direct.exists():
            return direct
        hits = list(root.glob("**/idx_kronos_all_15m.parquet"))
        if hits:
            return hits[0]
    raise FileNotFoundError("idx_kronos_all_15m.parquet tidak ditemukan. Clone repo ISTL ke /kaggle/working.")

DATA_PATH = find_data().resolve()
PROJECT_DIR = DATA_PATH.parent.parent
DAILY_PROJECT_DIR = PROJECT_DIR.parent / "Kronos IDX FineTune"
KRONOS_DIR = DAILY_PROJECT_DIR / "Kronos"

# Colab: mount Drive for persistent checkpoints when possible.
if Path("/content").exists():
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as exc:
            warnings.warn(f"Google Drive mount gagal; output Colab bersifat sementara: {exc}")
    RUNTIME_ROOT = (
        Path("/content/drive/MyDrive/ISTL-Kronos")
        if Path("/content/drive/MyDrive").exists() else Path("/content")
    )
elif Path("/kaggle/working").exists():
    RUNTIME_ROOT = Path("/kaggle/working")
else:
    RUNTIME_ROOT = PROJECT_DIR
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

KRONOS_COMMIT = "67b630e67f6a18c9e9be918d9b4337c960db1e9a"

if not (KRONOS_DIR / "model" / "kronos.py").exists():
    KRONOS_DIR = RUNTIME_ROOT / "Kronos"
    if not KRONOS_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/shiyu-coder/Kronos.git", str(KRONOS_DIR)], check=True)
    subprocess.run(["git", "-C", str(KRONOS_DIR), "checkout", KRONOS_COMMIT], check=True)

sys.path.insert(0, str(KRONOS_DIR))
from model import Kronos, KronosTokenizer, KronosPredictor

OUTPUT_DIR = RUNTIME_ROOT / "kronos_idx_15m_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "kronos_base_idx_all" / "best_model"
CHART_DIR = OUTPUT_DIR / "charts"
for p in [OUTPUT_DIR, CHECKPOINT_DIR, CHART_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cuda_info = {
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "compute_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    "compiled_arches": torch.cuda.get_arch_list() if torch.cuda.is_available() else [],
    "data": str(DATA_PATH),
    "kronos_source": str(KRONOS_DIR),
}
print(cuda_info)
print("Persistent output directory:", OUTPUT_DIR)
if DEVICE.type != "cuda":
    warnings.warn("GPU tidak aktif. Fine-tuning akan sangat lambat; aktifkan Kaggle GPU Accelerator.")
else:
    capability = torch.cuda.get_device_capability(0)
    supported_arches = set(torch.cuda.get_arch_list())
    expected_arch = f"sm_{capability[0]}{capability[1]}"
    if expected_arch not in supported_arches:
        raise RuntimeError(
            f"PyTorch {torch.__version__} tidak membawa kernel {expected_arch}. "
            "Install torch 2.7.1+cu128 dari sel pertama, restart kernel, lalu Run All."
        )
    try:
        smoke = torch.ones(8, device=DEVICE)
        torch.cuda.synchronize()
        del smoke
        torch.cuda.empty_cache()
        print("✓ CUDA compatibility smoke test passed; Blackwell SDPA enabled.")
    except RuntimeError as exc:
        raise RuntimeError(
            "CUDA binary tidak kompatibel dengan GPU Kaggle. Restart session "
            "dan jalankan notebook dari sel pertama; jangan hanya rerun sel ini."
        ) from exc

## 2. Configuration — 80 GB profile

Setiap epoch mengambil 200.000 window baru dari candidate pool dengan
probabilitas lebih tinggi untuk rezim terbaru. Validation Juli tidak
disampling. BF16, batch 128, TF32, dan worker paralel ditujukan untuk
A100/H100 80 GB; batch dapat dinaikkan ke 256 setelah benchmark.

In [ ]:
MODEL_ID = "NeoQuasar/Kronos-base"
BARS_PER_SESSION = 20
LOOKBACK = 240
PRED_LEN = 20
MAX_CONTEXT = 512
TRAIN_END = pd.Timestamp("2026-07-30 23:59:59")
VAL_START = pd.Timestamp("2026-07-01")
RECENT_START = pd.Timestamp("2026-06-01")
AS_OF_DATE = pd.Timestamp("2026-07-30 23:59:59")
MAX_STALE_CALENDAR_DAYS = 10

BATCH_SIZE = 128
EPOCHS = 4
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05
TRAIN_WINDOWS_PER_EPOCH = 200_000
REFIT_WINDOWS_PER_EPOCH = 200_000
RECENT_SAMPLE_WEIGHT = 3.0
# Notebook kernels and Python 3.12 can tear down forked persistent
# workers from a different PID. Start with the stable in-process
# loader; benchmark multiprocessing separately after correctness.
NUM_WORKERS = 0
MIN_ACTIVE_RATIO = 0.60
PATIENCE = 2

N_FORECAST_PATHS = 5
INFERENCE_ASSET_BATCH = 32
TEMPERATURE = 0.8
TOP_P = 0.9
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
print("Device:", DEVICE, "| AMP dtype:", AMP_DTYPE)

## 3. Load and audit the Parquet dataset

In [ ]:
raw = pd.read_parquet(DATA_PATH)
raw["date"] = pd.to_datetime(raw["date"])
if raw["date"].dt.tz is not None:
    raw["date"] = raw["date"].dt.tz_convert("Asia/Jakarta").dt.tz_localize(None)
universe_path = DATA_PATH.parent / "universe_all.csv"
universe = pd.read_csv(universe_path) if universe_path.exists() else pd.DataFrame({"ticker": sorted(raw["ticker"].unique())})
universe["ticker"] = universe["ticker"].astype(str).str.strip().str.upper()
TICKERS = universe["ticker"].drop_duplicates().tolist()
raw = raw[raw["ticker"].isin(TICKERS)].copy()
raw = raw.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"])

price_cols = ["open", "high", "low", "close"]
raw = raw.dropna(subset=price_cols)
raw["volume"] = raw["volume"].fillna(0).clip(lower=0)
raw["amount"] = raw.get("amount", raw[price_cols].mean(axis=1) * raw["volume"])
raw["amount"] = raw["amount"].fillna(raw[price_cols].mean(axis=1) * raw["volume"])

audit = (
    raw.groupby("ticker")
    .agg(rows=("date", "size"), start=("date", "min"), end=("date", "max"),
         zero_volume=("volume", lambda x: int((x <= 0).sum())))
    .reindex(TICKERS)
    .reset_index()
)
audit["active_pct"] = 1 - audit["zero_volume"] / audit["rows"]
display(audit)

fig = px.timeline(
    audit.sort_values("start"), x_start="start", x_end="end", y="ticker",
    color="active_pct", color_continuous_scale="Tealgrn",
    title="IDX Fine-Tuning Universe — Data Coverage"
)
fig.update_layout(template="plotly_white", height=900, coloraxis_colorbar_title="Active %")
fig.show()

print(f"{raw.ticker.nunique()} / {len(TICKERS)} tickers with data | {len(raw):,} rows | {raw.date.min()} → {raw.date.max()}")
print(f"Forecast anchor: close terakhir pada/sebelum {AS_OF_DATE.date()} (bar sesudah tanggal ini tidak dipakai).")

## 4. Multi-asset chronological dataset

Setiap sample dinormalisasi menggunakan mean/std **hanya dari lookback**.
Window yang terlalu banyak sesi volume nol tidak digunakan untuk training,
tetapi ticker tersebut tetap masuk tahap forecast.

In [ ]:
FEATURES = ["open", "high", "low", "close", "volume", "amount"]
TIME_FEATURES = ["minute", "hour", "weekday", "day", "month"]

class DynamicPanelKlineDataset(Dataset):
    def __init__(self, frame, split, samples_per_epoch=None):
        if split not in {"train", "val", "refit"}:
            raise ValueError("split must be train, val, or refit")
        self.split = split
        self.samples_per_epoch = samples_per_epoch
        self.series, self.candidates = {}, []
        window = LOOKBACK + PRED_LEN + 1

        for ticker, g in frame.groupby("ticker", sort=False):
            g = g.sort_values("date").reset_index(drop=True).copy()
            g["minute"] = g["date"].dt.minute
            g["hour"] = g["date"].dt.hour
            g["weekday"] = g["date"].dt.weekday
            g["day"] = g["date"].dt.day
            g["month"] = g["date"].dt.month
            self.series[ticker] = g

            for start in range(len(g) - window + 1):
                target_start = g.loc[start + LOOKBACK, "date"]
                target_end = g.loc[start + window - 1, "date"]
                active_ratio = (g.loc[start:start + LOOKBACK - 1, "volume"] > 0).mean()
                if active_ratio < MIN_ACTIVE_RATIO:
                    continue
                eligible = (
                    split == "train" and target_end < VAL_START
                    or split == "val" and target_start >= VAL_START and target_end <= TRAIN_END
                    or split == "refit" and target_end <= TRAIN_END
                )
                if eligible:
                    self.candidates.append((ticker, start, target_end))

        self.resample(0)
        print(f"{split}: {len(self.candidates):,} candidates; {len(self.indices):,} active windows")

    def resample(self, epoch):
        if not self.samples_per_epoch or len(self.candidates) <= self.samples_per_epoch:
            self.indices = [(ticker, start) for ticker, start, _ in self.candidates]
            return
        rng = np.random.default_rng(SEED + 10_000 * ({"train": 1, "refit": 2}.get(self.split, 0)) + epoch)
        weights = np.fromiter(
            (RECENT_SAMPLE_WEIGHT if target_end >= RECENT_START else 1.0 for _, _, target_end in self.candidates),
            dtype=np.float64,
        )
        weights /= weights.sum()
        chosen = rng.choice(len(self.candidates), self.samples_per_epoch, replace=False, p=weights)
        self.indices = [(self.candidates[i][0], self.candidates[i][1]) for i in chosen]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        ticker, start = self.indices[idx]
        window = self.series[ticker].iloc[start:start + LOOKBACK + PRED_LEN + 1]
        x = window[FEATURES].to_numpy(np.float32)
        stamps = window[TIME_FEATURES].to_numpy(np.float32)
        mean, std = x[:LOOKBACK].mean(axis=0), x[:LOOKBACK].std(axis=0)
        x = np.clip((x - mean) / (std + 1e-5), -5, 5)
        return torch.from_numpy(x), torch.from_numpy(stamps)

def make_loader(dataset, shuffle):
    kwargs = dict(
        dataset=dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=shuffle,
    )
    if NUM_WORKERS > 0:
        # Dynamic indices are refreshed each epoch, so workers must
        # not persist with a stale private copy of the dataset.
        kwargs.update(persistent_workers=False, prefetch_factor=4)
    return DataLoader(**kwargs)

train_ds = DynamicPanelKlineDataset(raw[raw.date <= TRAIN_END], "train", TRAIN_WINDOWS_PER_EPOCH)
val_ds = DynamicPanelKlineDataset(raw[raw.date <= TRAIN_END], "val")
if not len(train_ds) or not len(val_ds):
    raise RuntimeError("Train/validation windows kosong; periksa tanggal dan coverage data.")
val_loader = make_loader(val_ds, False)

## 5. Load pretrained Kronos-base

Tokenizer tidak diubah. Hanya bobot predictor 24.7M parameter yang
di-fine-tune agar token dynamics menyesuaikan karakter saham IDX.

In [ ]:
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base").to(DEVICE).eval()
model = Kronos.from_pretrained(MODEL_ID).to(DEVICE)
for p in tokenizer.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable predictor parameters: {trainable:,}")

## 6. Fine-tune predictor

In [ ]:
def causal_epoch(model, dataset, optimizer, scheduler, scaler, epoch, label):
    dataset.resample(epoch)
    loader = make_loader(dataset, True)
    model.train()
    total = 0.0
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(loader, desc=f"{label} epoch {epoch}", leave=False)
    for step, (batch_x, batch_stamp) in enumerate(progress, 1):
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_stamp = batch_stamp.to(DEVICE, non_blocking=True)
        with torch.no_grad():
            token_0, token_1 = tokenizer.encode(batch_x, half=True)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            logits = model(token_0[:, :-1], token_1[:, :-1], batch_stamp[:, :-1, :])
            loss, _, _ = model.head.compute_loss(
                logits[0], logits[1], token_0[:, 1:], token_1[:, 1:]
            )
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        total += loss.item()
        progress.set_postfix(loss=f"{total / step:.4f}")
    return total / len(loader)

@torch.no_grad()
def validation_loss(model):
    model.eval()
    total = 0.0
    for step, (batch_x, batch_stamp) in enumerate(tqdm(val_loader, desc="full July validation", leave=False), 1):
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_stamp = batch_stamp.to(DEVICE, non_blocking=True)
        token_0, token_1 = tokenizer.encode(batch_x, half=True)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            logits = model(token_0[:, :-1], token_1[:, :-1], batch_stamp[:, :-1, :])
            loss, _, _ = model.head.compute_loss(logits[0], logits[1], token_0[:, 1:], token_1[:, 1:])
        total += loss.item()
    return total / step

steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, fused=DEVICE.type == "cuda")
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: min(1.0, (step + 1) / warmup_steps) * 0.5 *
    (1.0 + math.cos(math.pi * max(0, step - warmup_steps) / max(1, total_steps - warmup_steps))),
)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)

history, best_val, best_epoch, stale_epochs = [], float("inf"), 0, 0
for epoch in range(1, EPOCHS + 1):
    train_loss = causal_epoch(model, train_ds, optimizer, scheduler, scaler, epoch, "domain adaptation")
    val_loss = validation_loss(model)
    row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "learning_rate": optimizer.param_groups[0]["lr"]}
    history.append(row)
    print(row)
    if val_loss < best_val:
        best_val, best_epoch, stale_epochs = val_loss, epoch, 0
        model.save_pretrained(CHECKPOINT_DIR)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"Selected epoch count: {best_epoch}; best July validation loss: {best_val:.6f}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.train_loss, mode="lines+markers", name="Train"))
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.val_loss, mode="lines+markers", name="Validation"))
fig.update_layout(
    title="Kronos-base Fine-Tuning Loss", xaxis_title="Epoch", yaxis_title="Token Cross-Entropy",
    template="plotly_white", height=430, hovermode="x unified"
)
fig.show()

## 7. Probabilistic forecast seluruh emiten

Model terbaik dimuat kembali. Lima sampled paths dibuat untuk setiap saham.
Seluruh path disimpan; ranking menggunakan expected 20-day return,
probabilitas return positif, peak return, dan downside percentile.

In [ ]:
# Clean production refit: restart from pretrained weights and train through 30 July.
# The epoch count was selected without allowing July into the selection-stage gradients.
refit_ds = DynamicPanelKlineDataset(raw[raw.date <= TRAIN_END], "refit", REFIT_WINDOWS_PER_EPOCH)
best_model = Kronos.from_pretrained(MODEL_ID).to(DEVICE)
refit_optimizer = torch.optim.AdamW(best_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, fused=DEVICE.type == "cuda")
refit_steps = math.ceil(len(refit_ds) / BATCH_SIZE) * best_epoch
refit_warmup = max(1, int(refit_steps * WARMUP_RATIO))
refit_scheduler = torch.optim.lr_scheduler.LambdaLR(
    refit_optimizer,
    lambda step: min(1.0, (step + 1) / refit_warmup) * 0.5 *
    (1.0 + math.cos(math.pi * max(0, step - refit_warmup) / max(1, refit_steps - refit_warmup))),
)
refit_scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)
refit_history = []
for epoch in range(1, best_epoch + 1):
    loss = causal_epoch(best_model, refit_ds, refit_optimizer, refit_scheduler, refit_scaler, epoch, "production refit")
    refit_history.append({"epoch": epoch, "train_loss": loss})
pd.DataFrame(refit_history).to_csv(OUTPUT_DIR / "production_refit_history.csv", index=False)
production_dir = OUTPUT_DIR / "production_model"
best_model.save_pretrained(production_dir)
predictor = KronosPredictor(best_model, tokenizer, device=str(DEVICE), max_context=MAX_CONTEXT)

contexts, x_times, y_times, valid_tickers, last_close_map, skipped = [], [], [], [], {}, []
# Reuse the most common observed intraday clock. Twenty bars normally
# span one IDX session; Friday/session anomalies remain data-driven.
session_clock = (
    raw.assign(clock=raw["date"].dt.strftime("%H:%M"))
    .groupby([raw["date"].dt.date, "clock"]).size().reset_index(name="n")
    .groupby("clock")["n"].count().sort_values(ascending=False)
)
clock_values = sorted(session_clock.head(BARS_PER_SESSION).index)
future_values, future_day = [], (AS_OF_DATE + pd.offsets.BDay(1)).normalize()
while len(future_values) < PRED_LEN:
    if future_day.weekday() < 5:
        future_values.extend(
            future_day + pd.Timedelta(hours=int(clock[:2]), minutes=int(clock[3:]))
            for clock in clock_values
        )
    future_day += pd.offsets.BDay(1)
global_future_dates = pd.Series(future_values[:PRED_LEN])
for ticker in TICKERS:
    g = raw[(raw.ticker.eq(ticker)) & (raw.date <= AS_OF_DATE)].sort_values("date").tail(LOOKBACK).copy()
    if len(g) < LOOKBACK:
        skipped.append({"ticker": ticker, "reason": "insufficient_history", "bars": len(g), "last_date": g.date.max() if len(g) else pd.NaT})
        continue
    stale_days = int((AS_OF_DATE - g["date"].max()).days)
    if stale_days > MAX_STALE_CALENDAR_DAYS:
        skipped.append({"ticker": ticker, "reason": "stale_or_suspended", "bars": len(g), "last_date": g.date.max()})
        continue
    x_df = g[FEATURES].copy()
    contexts.append(x_df)
    x_times.append(pd.Series(pd.to_datetime(g["date"]).to_numpy()))
    y_times.append(global_future_dates.copy())
    valid_tickers.append(ticker)
    last_close_map[ticker] = float(g["close"].iloc[-1])
skipped_df = pd.DataFrame(skipped)
skipped_df.to_csv(OUTPUT_DIR / "skipped_tickers.csv", index=False)
print(f"Eligible forecast: {len(valid_tickers)} / {len(TICKERS)} | skipped: {len(skipped_df)}")

In [ ]:
if not valid_tickers:
    raise RuntimeError("Tidak ada ticker eligible untuk forecast.")

all_paths = []
for path_id in range(N_FORECAST_PATHS):
    torch.manual_seed(SEED + 10_000 + path_id)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 10_000 + path_id)
    for start in tqdm(range(0, len(valid_tickers), INFERENCE_ASSET_BATCH), desc=f"Forecast path {path_id+1}"):
        stop = start + INFERENCE_ASSET_BATCH
        preds = predictor.predict_batch(
            df_list=contexts[start:stop],
            x_timestamp_list=x_times[start:stop],
            y_timestamp_list=y_times[start:stop],
            pred_len=PRED_LEN,
            T=TEMPERATURE,
            top_p=TOP_P,
            top_k=0,
            sample_count=1,
            verbose=False,
        )
        for ticker, pred in zip(valid_tickers[start:stop], preds):
            path = pred.reset_index().rename(columns={"index": "date"})
            path["ticker"] = ticker
            path["path_id"] = path_id
            path["horizon_bar"] = np.arange(1, len(path) + 1)
            all_paths.append(path)

if not all_paths:
    raise RuntimeError("Predictor tidak menghasilkan forecast path.")
forecasts = pd.concat(all_paths, ignore_index=True)
forecasts["date"] = pd.to_datetime(forecasts["date"])
forecasts.to_parquet(OUTPUT_DIR / "all_forecast_paths.parquet", index=False)
print(f"Saved {len(forecasts):,} forecast rows for {forecasts.ticker.nunique()} tickers.")

## 8. Rank all stocks and display the 30 strongest expected returns

In [ ]:
final_day = forecasts[forecasts.horizon_bar.eq(PRED_LEN)].copy()
peak_by_path = forecasts.groupby(["ticker", "path_id"])["close"].max().rename("peak_close").reset_index()
final_by_path = final_day[["ticker", "path_id", "close"]].rename(columns={"close": "final_close"})
path_stats = final_by_path.merge(peak_by_path, on=["ticker", "path_id"])
path_stats["last_close"] = path_stats["ticker"].map(last_close_map)
path_stats["return_20bar"] = path_stats["final_close"] / path_stats["last_close"] - 1
path_stats["peak_return_20bar"] = path_stats["peak_close"] / path_stats["last_close"] - 1

ranking = (
    path_stats.groupby("ticker")
    .agg(
        last_close=("last_close", "first"),
        expected_close_20bar=("final_close", "mean"),
        expected_return_20bar=("return_20bar", "mean"),
        median_return_20bar=("return_20bar", "median"),
        probability_up=("return_20bar", lambda x: float((x > 0).mean())),
        downside_p10=("return_20bar", lambda x: float(np.quantile(x, 0.10))),
        expected_peak_return_20bar=("peak_return_20bar", "mean"),
        forecast_dispersion=("return_20bar", "std"),
    )
    .reset_index()
    .sort_values(["expected_return_20bar", "probability_up"], ascending=False)
)
ranking["predicted_up"] = (
    (ranking["expected_return_20bar"] > 0) &
    (ranking["probability_up"] >= 0.60)
)
ranking["rank"] = np.arange(1, len(ranking) + 1)
ranking.to_csv(OUTPUT_DIR / "all_ticker_ranking.csv", index=False)
ranking.to_parquet(OUTPUT_DIR / "all_ticker_ranking.parquet", index=False)

top30 = ranking[ranking["predicted_up"]].head(30).copy()
if len(top30) < 30:
    print(f"Model hanya menemukan {len(top30)} ticker dengan expected return > 0 dan P(up) ≥ 60%; hasil tidak dipaksakan menjadi 30.")
display(
    top30.style
    .format({
        "last_close": "{:,.0f}", "expected_close_20bar": "{:,.0f}",
        "expected_return_20bar": "{:+.2%}", "median_return_20bar": "{:+.2%}",
        "probability_up": "{:.0%}", "downside_p10": "{:+.2%}",
        "expected_peak_return_20bar": "{:+.2%}", "forecast_dispersion": "{:.2%}",
    })
    .background_gradient(subset=["expected_return_20bar"], cmap="RdYlGn")
    .background_gradient(subset=["probability_up"], cmap="Blues")
)

In [ ]:
chart = top30.sort_values("expected_return_20bar")
colors = chart["probability_up"]
fig = go.Figure(go.Bar(
    x=chart["expected_return_20bar"], y=chart["ticker"], orientation="h",
    marker=dict(color=colors, colorscale="Tealgrn", cmin=0, cmax=1,
                colorbar=dict(title="P(Return > 0)")),
    customdata=np.c_[chart["probability_up"], chart["downside_p10"]],
    hovertemplate="<b>%{y}</b><br>Expected return: %{x:.2%}<br>P(up): %{customdata[0]:.0%}<br>Downside P10: %{customdata[1]:.2%}<extra></extra>"
))
fig.add_vline(x=0, line_dash="dash", line_color="#64748b")
fig.update_layout(
    title="Kronos-base — Top 30 Expected 20-Day Returns",
    xaxis_tickformat=".1%", xaxis_title="Expected return", yaxis_title="",
    template="plotly_white", height=850, margin=dict(l=70, r=40, t=80, b=60)
)
fig.write_html(CHART_DIR / "top30_expected_returns.html")
fig.show()

## 9. Positive picks for each of the next five sessions

Return Day 1 memakai close aktual terakhir pada/sebelum `AS_OF_DATE`.
Return Day 2 memakai predicted close Day 1 pada path yang sama, dan
seterusnya. Rasio dihitung per stochastic path sebelum dirata-ratakan.

In [ ]:
first_5 = (
    forecasts[forecasts["horizon_bar"].between(1, 5)]
    .sort_values(["ticker", "path_id", "horizon_bar"])
    .copy()
)
first_5["last_actual_close"] = first_5["ticker"].map(last_close_map)
first_5["previous_close"] = (
    first_5.groupby(["ticker", "path_id"])["close"].shift(1)
    .fillna(first_5["last_actual_close"])
)
first_5["daily_return"] = first_5["close"] / first_5["previous_close"] - 1

daily_ranking = (
    first_5.groupby(["horizon_bar", "date", "ticker"])
    .agg(
        previous_expected_close=("previous_close", "mean"),
        expected_close=("close", "mean"),
        expected_daily_return=("daily_return", "mean"),
        median_daily_return=("daily_return", "median"),
        probability_daily_up=("daily_return", lambda x: float((x > 0).mean())),
        downside_p10=("daily_return", lambda x: float(np.quantile(x, 0.10))),
        upside_p90=("daily_return", lambda x: float(np.quantile(x, 0.90))),
    )
    .reset_index()
)
positive_each_day = daily_ranking[
    (daily_ranking["expected_daily_return"] > 0) &
    (daily_ranking["probability_daily_up"] >= 0.60)
].copy()
positive_each_day["rank"] = (
    positive_each_day.groupby("horizon_bar")["expected_daily_return"]
    .rank(method="first", ascending=False).astype(int)
)
positive_each_day = positive_each_day.sort_values(["horizon_bar", "rank"])
daily_ranking.to_parquet(OUTPUT_DIR / "all_daily_rankings_bar1_to_bar5.parquet", index=False)
positive_each_day.to_csv(OUTPUT_DIR / "positive_picks_bar1_to_bar5.csv", index=False)

for day in range(1, 6):
    result = positive_each_day[positive_each_day.horizon_bar.eq(day)].head(30)
    forecast_date = result["date"].iloc[0].date() if len(result) else global_future_dates.iloc[day-1].date()
    print(f"BAR {day} ({forecast_date}) — {len(result)} positive candidates shown")
    display(result.style.format({
        "previous_expected_close": "{:,.0f}", "expected_close": "{:,.0f}",
        "expected_daily_return": "{:+.2%}", "median_daily_return": "{:+.2%}",
        "probability_daily_up": "{:.0%}", "downside_p10": "{:+.2%}",
        "upside_p90": "{:+.2%}",
    }).background_gradient(subset=["expected_daily_return"], cmap="RdYlGn"))

In [ ]:
heatmap_names = (
    positive_each_day.groupby("ticker")["expected_daily_return"].mean()
    .nlargest(30).index
)
heatmap = (
    daily_ranking[daily_ranking.ticker.isin(heatmap_names)]
    .pivot(index="ticker", columns="horizon_bar", values="expected_daily_return")
    .reindex(heatmap_names)
)
fig = px.imshow(
    heatmap, aspect="auto", color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0, text_auto=".1%",
    labels={"x": "Forecast bar", "y": "Ticker", "color": "Daily return"},
    title=f"Expected Daily Return — Anchor Close {AS_OF_DATE.date()}"
)
fig.update_layout(template="plotly_white", height=850)
fig.write_html(CHART_DIR / "daily_return_heatmap_bar1_to_bar5.html")
fig.show()

## 10. Professional dashboards for the top five stocks

In [ ]:
top5_pool = top30 if len(top30) >= 5 else ranking[ranking["expected_return_20bar"] > 0]
top5 = top5_pool.head(5)["ticker"].tolist()
print("Top 5:", top5)

for ticker in top5:
    hist = raw[(raw.ticker.eq(ticker)) & (raw.date <= AS_OF_DATE)].sort_values("date").tail(70)
    fc = forecasts[forecasts.ticker.eq(ticker)]
    band = (
        fc.groupby("date")
        .agg(
            mean_close=("close", "mean"),
            p10_close=("close", lambda x: np.quantile(x, 0.10)),
            p90_close=("close", lambda x: np.quantile(x, 0.90)),
            mean_volume=("volume", "mean"),
        )
        .reset_index()
    )
    stats = ranking.set_index("ticker").loc[ticker]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        row_heights=[0.72, 0.28],
        subplot_titles=[
            f"{ticker} — expected {stats.expected_return_20bar:+.2%} | P(up) {stats.probability_up:.0%}",
            "Historical and Forecast Volume"
        ]
    )
    fig.add_trace(go.Candlestick(
        x=hist.date, open=hist.open, high=hist.high, low=hist.low, close=hist.close,
        name="Historical OHLC", increasing_line_color="#059669", decreasing_line_color="#dc2626"
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=band.date, y=band.p90_close, line=dict(width=0), showlegend=False,
        hoverinfo="skip", name="P90"
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=band.date, y=band.p10_close, line=dict(width=0), fill="tonexty",
        fillcolor="rgba(14,116,144,0.16)", name="80% forecast interval",
        hovertemplate="P10: %{y:,.0f}<extra></extra>"
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=band.date, y=band.mean_close, mode="lines+markers",
        line=dict(color="#0e7490", width=3), marker=dict(size=4),
        name="Mean forecast", hovertemplate="%{x|%d %b %Y}<br>%{y:,.0f}<extra></extra>"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=hist.date, y=hist.volume, marker_color="#94a3b8", name="Historical volume"
    ), row=2, col=1)
    fig.add_trace(go.Bar(
        x=band.date, y=band.mean_volume, marker_color="#0e7490", name="Forecast volume"
    ), row=2, col=1)

    fig.update_layout(
        template="plotly_white", height=720,
        title=dict(text=f"Kronos IDX Forecast Dashboard — {ticker}", x=0.5),
        legend=dict(orientation="h", y=1.03, x=0),
        xaxis_rangeslider_visible=False, hovermode="x unified",
        margin=dict(l=60, r=40, t=105, b=50)
    )
    fig.update_yaxes(title_text="Price (IDR)", tickformat=",", row=1, col=1)
    fig.update_yaxes(title_text="Volume", tickformat=".2s", row=2, col=1)
    fig.write_html(CHART_DIR / f"{ticker}_forecast_dashboard.html")
    fig.show()

## 11. Package Kaggle outputs

In [ ]:
metadata = {
    "kronos_commit": KRONOS_COMMIT,
    "pretrained_model": MODEL_ID,
    "training_profile": "80gb_15m_dynamic_refit",
    "selected_epochs": int(best_epoch),
    "train_windows_per_epoch": TRAIN_WINDOWS_PER_EPOCH,
    "refit_windows_per_epoch": REFIT_WINDOWS_PER_EPOCH,
    "pretrained_tokenizer": "NeoQuasar/Kronos-Tokenizer-base",
    "training_target_end_exclusive": str(VAL_START.date()),
    "validation_start": str(VAL_START.date()),
    "validation_end": str(TRAIN_END.date()),
    "forecast_as_of_date": str(AS_OF_DATE.date()),
    "interval": "15m",
    "bars_per_session_assumption": BARS_PER_SESSION,
    "lookback": LOOKBACK,
    "prediction_horizon": PRED_LEN,
    "forecast_paths": N_FORECAST_PATHS,
    "tickers_requested": TICKERS,
    "tickers_forecast": valid_tickers,
    "best_validation_loss": float(best_val),
    "warning": "Statistical forecast, not investment advice."
}
with open(OUTPUT_DIR / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Download archive:", archive)
print("\nFiles:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR))